# Project01: Large Movie Review Dataset  
binary classification task for language

**Keras Applications**  
https://keras.io/api/applications/

## Data prepration

In [1]:
# Import data
from pathlib import Path
import tarfile

tar_path = Path("aclImdb_v1.tar.gz")
data_dir = Path("aclImdb")

# Download only if the tar.gz file doesn't already exist
if not tar_path.exists():
    !wget -O {tar_path} http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
else:
    print(f"{tar_path} already exists, skipping download.")

# Extract only if the extracted folder doesn't already exist
if not data_dir.exists():
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall()
    print(f"Extracted to {data_dir}")
else:
    print(f"{data_dir} already exists, skipping extract.")

aclImdb_v1.tar.gz already exists, skipping download.
aclImdb already exists, skipping extract.


In [2]:
# See some random data
from pathlib import Path
import random

base = Path("aclImdb")  # change if needed

def show_random(split="train", label=None, max_chars=800):
    if label is None:
        label = random.choice(["pos", "neg"])
    files = list((base / split / label).glob("*.txt"))
    path = random.choice(files)
    text = path.read_text(encoding="utf-8")

    print(f"{split=} {label=} {path.name}")
    print("-" * 60)
    print(text[:max_chars])

show_random()  # random train review, random label

split='train' label='pos' 4976_10.txt
------------------------------------------------------------
A Frank Capra WONDERS OF LIFE film.<br /><br />Keeping the blood pumping through our veins is the responsibility of hardworking HEMO THE Magnificent.<br /><br />In the mid-1950's, AT&T and Bell Science teamed with famed Hollywood director Frank Capra to produce a series of CBS television science films to educate the public about the Universe around them. A far cry from the dreary black & white fodder so often foisted off on young scholars, the Capra films would both instruct and entertain with lively scripts and eye-catching visuals shown in Technicolor. The four films - OUR MR. SUN (1956), THE STRANGE CASE OF THE COSMIC RAYS (1957), HEMO THE MAGNIFICENT (1957), THE UNCHAINED GODDESS (1958) - quickly became schoolhouse favorites, where they were endlessly shown in 16mm format.<br /><br />T


In [3]:
from pathlib import Path

base = Path("aclImdb")  # adjust if needed

def count_txt(path: Path) -> int:
    return len(list(path.glob("*.txt")))

for split in ["train", "test"]:
    for label in ["pos", "neg"]:
        folder = base / split / label
        print(split, label, count_txt(folder))

train pos 12500
train neg 12500
test pos 12500
test neg 12500


In [4]:
# 1) Load IMDB files into memory
from pathlib import Path

base = Path("aclImdb")          # adjust if needed
train_dir = base / "train"
test_dir  = base / "test"

# ---- load TRAIN files ----
X_train = []
y_train = []

for label in ["pos", "neg"]:
    folder = train_dir / label
    for path in folder.glob("*.txt"):
        X_train.append(path.read_text(encoding="utf-8"))
        y_train.append(1 if label == "pos" else 0)

# ---- load TEST files ----
texts_test = []
labels_test = []

for label in ["pos", "neg"]:
    folder = test_dir / label
    for path in folder.glob("*.txt"):
        texts_test.append(path.read_text(encoding="utf-8"))
        labels_test.append(1 if label == "pos" else 0)
print(len(X_train), len(texts_test))

25000 25000


In [5]:
#2) Split into 15k validation and 10k test

from sklearn.model_selection import train_test_split

X_val, X_test, y_val, y_test= train_test_split(
    texts_test,
    labels_test,
    train_size=15_000,
    test_size=10_000,
    random_state=42,
    stratify=labels_test,   # preserves class balance
)

print(len(X_val), len(X_test))  

15000 10000


In [6]:
import tensorflow as tf

BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE
MAX_FEATURES = 20000
SEQ_LEN = 300

# tf.data
ds_train = tf.data.Dataset.from_tensor_slices((X_train, y_train))
ds_val   = tf.data.Dataset.from_tensor_slices((X_val, y_val))
ds_test  = tf.data.Dataset.from_tensor_slices((X_test, y_test))

ds_train = ds_train.shuffle(len(X_train)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val   = ds_val.batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test  = ds_test.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# vectorization
vectorize = tf.keras.layers.TextVectorization(
    max_tokens=MAX_FEATURES,
    output_mode="int",
    output_sequence_length=SEQ_LEN,
)

text_only_train = ds_train.map(lambda x, y: x)
vectorize.adapt(text_only_train)

# model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,), dtype=tf.string),
    vectorize,
    tf.keras.layers.Embedding(MAX_FEATURES, 128),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

import datetime as dt
# callbacks
log_dir = "Tensorboard_logs/model_1/" + dt.datetime.now().strftime("%Y%m%d-%H%M")

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

tensorboard_cb = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)
model.summary()

history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=100,
    callbacks=[early_stopping, tensorboard_cb]
)

test_loss, test_acc = model.evaluate(ds_test)
print("Test accuracy:", test_acc)

I0000 00:00:1777017318.745416 2335061 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777017318.775458 2335061 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777017319.430186 2335061 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1777017319.892348 2335061 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries men

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 300)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 300, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,568,321 (9.80 MB)

 Trainable params: 2,568,321 (9.80 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.7166 - loss: 0.5403 - val_accuracy: 0.7407 - val_loss: 0.4897
Epoch 2/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.8676 - loss: 0.3148 - val_accuracy: 0.8600 - val_loss: 0.3552
Epoch 3/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9040 - loss: 0.2394 - val_accuracy: 0.8282 - val_loss: 0.3886
Epoch 4/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9227 - loss: 0.2012 - val_accuracy: 0.8677 - val_loss: 0.3261
Epoch 5/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9364 - loss: 0.1725 - val_accuracy: 0.8485 - val_loss: 0.3830
Epoch 6/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9443 - loss: 0.1515 - val_accuracy: 0.7805 - val_loss: 0.6465
Epoch 7/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9440 - loss: 0.1488 - val_accuracy: 0.8681 - val_loss: 0.3781
Epoch 8/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9614 - loss: 0.1156 - 

In [7]:
import tensorflow as tf

BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE
MAX_FEATURES = 20000
SEQ_LEN = 300

# tf.data
ds_train = tf.data.Dataset.from_tensor_slices((X_train, y_train))
ds_val   = tf.data.Dataset.from_tensor_slices((X_val, y_val))
ds_test  = tf.data.Dataset.from_tensor_slices((X_test, y_test))

ds_train = ds_train.shuffle(len(X_train)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val   = ds_val.batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test  = ds_test.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# vectorization
vectorize = tf.keras.layers.TextVectorization(
    max_tokens=MAX_FEATURES,
    output_mode="int",
    output_sequence_length=SEQ_LEN,
)

text_only_train = ds_train.map(lambda x, y: x)
vectorize.adapt(text_only_train)

# model
model_2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,), dtype=tf.string),
    vectorize,
    tf.keras.layers.Embedding(MAX_FEATURES, 64),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation="relu",kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model_2.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

import datetime as dt
# callbacks
log_dir = "Tensorboard_logs/model_2/" + dt.datetime.now().strftime("%Y%m%d-%H%M")

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

tensorboard_cb = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)
model_2.summary()

history_2 = model_2.fit(
    ds_train,
    validation_data=ds_val,
    epochs=100,
    callbacks=[early_stopping, tensorboard_cb]
)

test_loss, test_acc = model_2.evaluate(ds_test)
print("Test accuracy:", test_acc)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 300)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 300, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,282,113 (4.89 MB)

 Trainable params: 1,282,113 (4.89 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.6910 - loss: 0.5867 - val_accuracy: 0.8170 - val_loss: 0.4311
Epoch 2/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8552 - loss: 0.3588 - val_accuracy: 0.8589 - val_loss: 0.3484
Epoch 3/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.8953 - loss: 0.2879 - val_accuracy: 0.8751 - val_loss: 0.3209
Epoch 4/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9151 - loss: 0.2456 - val_accuracy: 0.8757 - val_loss: 0.3234
Epoch 5/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9285 - loss: 0.2177 - val_accuracy: 0.8635 - val_loss: 0.3440
Epoch 6/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9392 - loss: 0.1920 - val_accuracy: 0.8587 - val_loss: 0.3691
Epoch 7/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9389 - loss: 0.1920 - val_accuracy: 0.8633 - val_loss: 0.3648
Epoch 8/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9455 - loss: 0.1768 - val_accu

In [8]:
import tensorflow as tf

BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE
MAX_FEATURES = 20000
SEQ_LEN = 300

# tf.data
ds_train = tf.data.Dataset.from_tensor_slices((X_train, y_train))
ds_val   = tf.data.Dataset.from_tensor_slices((X_val, y_val))
ds_test  = tf.data.Dataset.from_tensor_slices((X_test, y_test))

ds_train = ds_train.shuffle(len(X_train)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val   = ds_val.batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test  = ds_test.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# vectorization
vectorize = tf.keras.layers.TextVectorization(
    max_tokens=MAX_FEATURES,
    output_mode="int",
    output_sequence_length=SEQ_LEN,
)

text_only_train = ds_train.map(lambda x, y: x)
vectorize.adapt(text_only_train)

# model
inputs = tf.keras.Input(shape=(1,), dtype=tf.string)
x = vectorize(inputs)
x = tf.keras.layers.Embedding(MAX_FEATURES, 128)(x)
x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(64, return_sequences=True))(x)
x = tf.keras.layers.GlobalMaxPooling1D()(x)
x = tf.keras.layers.Dropout(0.5)(x)
x = tf.keras.layers.Dense(64, activation="relu")(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model_3 = tf.keras.Model(inputs, outputs)

model_3.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

import datetime as dt
# callbacks
log_dir = "Tensorboard_logs/model_3/" + dt.datetime.now().strftime("%Y%m%d-%H%M")

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

tensorboard_cb = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)
model_3.summary()

history_3 = model_3.fit(
    ds_train,
    validation_data=ds_val,
    epochs=100,
    callbacks=[early_stopping, tensorboard_cb]
)

test_loss, test_acc = model_3.evaluate(ds_test)
print("Test accuracy:", test_acc)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_2            │ (None, 300)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 300, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 300, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,667,137 (10.17 MB)

 Trainable params: 2,667,137 (10.17 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 37s 91ms/step - accuracy: 0.7472 - loss: 0.4894 - val_accuracy: 0.8694 - val_loss: 0.3079
Epoch 2/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 35s 91ms/step - accuracy: 0.9073 - loss: 0.2455 - val_accuracy: 0.8501 - val_loss: 0.3532
Epoch 3/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 35s 90ms/step - accuracy: 0.9459 - loss: 0.1494 - val_accuracy: 0.8561 - val_loss: 0.3658
Epoch 4/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 35s 91ms/step - accuracy: 0.9690 - loss: 0.0907 - val_accuracy: 0.8531 - val_loss: 0.4472
Epoch 5/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 36s 91ms/step - accuracy: 0.9772 - loss: 0.0673 - val_accuracy: 0.8423 - val_loss: 0.5678
Epoch 6/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 35s 91ms/step - accuracy: 0.9817 - loss: 0.0544 - val_accuracy: 0.8473 - val_loss: 0.5121
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 1.
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.8757 - loss: 0.2957
Test accuracy: 0.8756999969482422


In [10]:
import tensorflow as tf

BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE
MAX_FEATURES = 20000
SEQ_LEN = 300

# tf.data
ds_train = tf.data.Dataset.from_tensor_slices((X_train, y_train))
ds_val   = tf.data.Dataset.from_tensor_slices((X_val, y_val))
ds_test  = tf.data.Dataset.from_tensor_slices((X_test, y_test))

ds_train = ds_train.shuffle(len(X_train)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val   = ds_val.batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test  = ds_test.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# vectorization
vectorize = tf.keras.layers.TextVectorization(
    max_tokens=MAX_FEATURES,
    output_mode="int",
    output_sequence_length=SEQ_LEN,
)

text_only_train = ds_train.map(lambda x, y: x)
vectorize.adapt(text_only_train)

# model
inputs = tf.keras.Input(shape=(1,), dtype=tf.string)
x = vectorize(inputs)  # same TextVectorization
x = tf.keras.layers.Embedding(MAX_FEATURES, 128)(x)

conv_outputs = []
for k in [3, 4, 5]:
    c = tf.keras.layers.Conv1D(128, k, activation="relu")(x)
    c = tf.keras.layers.GlobalMaxPooling1D()(c)
    conv_outputs.append(c)

x = tf.keras.layers.Concatenate()(conv_outputs)
x = tf.keras.layers.Dropout(0.5)(x)
x = tf.keras.layers.Dense(64, activation="relu")(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model_4 = tf.keras.Model(inputs, outputs)

model_4.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

import datetime as dt
# callbacks
log_dir = "Tensorboard_logs/model_4/" + dt.datetime.now().strftime("%Y%m%d-%H%M")

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

tensorboard_cb = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)
model_4.summary()

history_4 = model_4.fit(
    ds_train,
    validation_data=ds_val,
    epochs=100,
    callbacks=[early_stopping, tensorboard_cb]
)

test_loss, test_acc = model_4.evaluate(ds_test)
print("Test accuracy:", test_acc)

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 300)       │          0 │ input_layer_3[0]… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 300, 128)  │  2,560,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 298, 128)  │     49,280 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 297, 128)  │     65,664 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 296, 128)  │     82,048 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 384)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 384)       │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │     24,640 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 64)        │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1)         │         65 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,781,697 (10.61 MB)

 Trainable params: 2,781,697 (10.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 47ms/step - accuracy: 0.7161 - loss: 0.5248 - val_accuracy: 0.8569 - val_loss: 0.3355
Epoch 2/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 18s 47ms/step - accuracy: 0.8828 - loss: 0.2904 - val_accuracy: 0.8789 - val_loss: 0.2872
Epoch 3/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 18s 47ms/step - accuracy: 0.9410 - loss: 0.1604 - val_accuracy: 0.8710 - val_loss: 0.3281
Epoch 4/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 18s 47ms/step - accuracy: 0.9737 - loss: 0.0806 - val_accuracy: 0.8649 - val_loss: 0.4033
Epoch 5/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 48ms/step - accuracy: 0.9853 - loss: 0.0444 - val_accuracy: 0.8563 - val_loss: 0.4903
Epoch 6/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 48ms/step - accuracy: 0.9909 - loss: 0.0291 - val_accuracy: 0.8539 - val_loss: 0.6136
Epoch 7/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 18s 47ms/step - accuracy: 0.9929 - loss: 0.0217 - val_accuracy: 0.8548 - val_loss: 0.6233
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 2.


In [12]:
import tensorflow as tf
import tensorflow_hub as hub
import datetime as dt


BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE


ds_train = tf.data.Dataset.from_tensor_slices((X_train, y_train))
ds_val   = tf.data.Dataset.from_tensor_slices((X_val, y_val))
ds_test  = tf.data.Dataset.from_tensor_slices((X_test, y_test))

ds_train = ds_train.cache().shuffle(len(X_train)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val   = ds_val.cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test  = ds_test.cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)


encoder = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")


class USELayer(tf.keras.layers.Layer):
    def __init__(self, encoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder

    def call(self, inputs):
        return self.encoder(inputs)


inputs = tf.keras.Input(shape=(), dtype=tf.string)
x = USELayer(encoder)(inputs)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model_5 = tf.keras.Model(inputs, outputs)

model_5.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


log_dir = "Tensorboard_logs/model_5/" + dt.datetime.now().strftime("%Y%m%d-%H%M")

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

tensorboard_cb = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

model_5.summary()

history_5 = model_5.fit(
    ds_train,
    validation_data=ds_val,
    epochs=100,
    callbacks=[early_stopping, tensorboard_cb]
)

test_loss, test_acc = model_5.evaluate(ds_test)
print("Test accuracy:", test_acc)

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None)                 │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ use_layer (USELayer)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,793 (257.00 KB)

 Trainable params: 65,793 (257.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100


/home/paminidigehsara/Desktop/controlEx_prep/.venv_ce/lib/python3.12/site-packages/keras/src/callbacks/tensorboard.py:680: UserWarning: Model failed to serialize as JSON. Ignoring... 
Object USELayer was created by passing
non-serializable argument values in `__init__()`,
and therefore the object must override `get_config()` in
order to be serializable. Please implement `get_config()`.

Example:


class CustomLayer(keras.layers.Layer):
    def __init__(self, arg1, arg2, **kwargs):
        super().__init__(**kwargs)
        self.arg1 = arg1
        self.arg2 = arg2

    def get_config(self):
        config = super().get_config()
        config.update({
            "arg1": self.arg1,
            "arg2": self.arg2,
        })
        return config

  warnings.warn(f"Model failed to serialize as JSON. Ignoring... {exc}")


  3/391 ━━━━━━━━━━━━━━━━━━━━ 12s 31ms/step - accuracy: 0.5113 - loss: 0.6941

E0000 00:00:1777018060.010832 2335061 util.cc:131] oneDNN supports DT_INT64 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


391/391 ━━━━━━━━━━━━━━━━━━━━ 21s 51ms/step - accuracy: 0.8089 - loss: 0.4344 - val_accuracy: 0.8522 - val_loss: 0.3412
Epoch 2/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 51ms/step - accuracy: 0.8378 - loss: 0.3671 - val_accuracy: 0.8533 - val_loss: 0.3362
Epoch 3/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 51ms/step - accuracy: 0.8411 - loss: 0.3618 - val_accuracy: 0.8555 - val_loss: 0.3310
Epoch 4/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 50ms/step - accuracy: 0.8434 - loss: 0.3538 - val_accuracy: 0.8566 - val_loss: 0.3290
Epoch 5/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 51ms/step - accuracy: 0.8426 - loss: 0.3542 - val_accuracy: 0.8588 - val_loss: 0.3258
Epoch 6/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 51ms/step - accuracy: 0.8494 - loss: 0.3474 - val_accuracy: 0.8580 - val_loss: 0.3252
Epoch 7/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 50ms/step - accuracy: 0.8450 - loss: 0.3529 - val_accuracy: 0.8551 - val_loss: 0.3278
Epoch 8/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 50ms/step - accuracy: 0.8476 - loss: 0.3482 - val_

In [9]:
# %load_ext tensorboard
# %tensorboard --logdir logs/fit